# Load SEC filings for your action

Choose an action and dates, download the filings, and inspect the text in a pandas table. Run the cells from top to bottom. This notebook does not call an LLM.


In [6]:
from pathlib import Path
import sys
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks' / 'evaluation' / 'load_action_filings.py').exists())
sys.path.insert(0, str(ROOT / 'notebooks' / 'evaluation'))
from load_action_filings import ACTIONS, load_action
pd.DataFrame([{'action': action, **spec} for action, spec in ACTIONS.items()])


,action,primary,supporting,conditional,query
0,tender_offer,"[SC TO-T, SC TO-T/A, SC TO-I, SC TO-I/A]","[SC 14D9, SC 14D9/A]",[],"""tender offer"" OR ""offer to purchase"""
1,exchange_offer,"[SC TO-T, SC TO-T/A, SC TO-I, SC TO-I/A, S-4, ...","[SC 14D9, SC 14D9/A, SC TO-C, 425, 8-K, 6-K]","[SC 13E3, SC 13E3/A]","""exchange offer"" OR ""offer to exchange"""
2,rights_issue,"[424B2, 424B3, 424B5, S-3, S-3ASR]","[8-K, 424B4]",[],"""rights offering"" OR ""subscription rights"" OR ..."
3,merger,"[DEFM14A, PREM14A, S-4, S-4/A, SC 13E3, SC 13E...","[8-K, 425]",[],"""merger"" OR ""business combination"""
4,conversion,"[8-K, 424B2, 424B3, S-3, S-3ASR]","[10-K, 10-Q]",[],"""conversion rate"" OR ""conversion price"" OR ""ma..."


## 1. Choose what to download

Edit the settings below. Dates are filing dates; the count is a maximum number of filings, not events.

Before running, set `SEC_USER_AGENT` to your name/project and contact email in the environment running Jupyter. No SEC API key or Chrome is needed.


In [7]:
ACTION = 'exchange_offer'  # tender_offer, exchange_offer, rights_issue, merger, conversion
START = '2023-07-01'
END = '2023-08-31'
N_FILINGS = None  # None loads every match; use a number for a smaller sample
OUTPUT = ROOT / 'notebooks' / 'evaluation' / 'data' / f'{ACTION}_{START}_{END}'
# Optional: use a saved filing list instead of searching SEC.
METADATA_PATH = None  # ROOT / 'data/processed/documents.parquet'


## 2. Download the files

This cell contacts SEC and reuses files already saved. The forms and keywords come from `corporate_actions.json`.

With `N_FILINGS = None`, all search matches are loaded. Set `N_FILINGS` to a number for a smaller diversified sample. Check `search_counts.csv` for SEC-reported totals and any limits reached.


In [8]:
# set env variable for SEC_USER_AGENT="Name email@gmail.com"
filings_df = load_action(
    action=ACTION, start=START, end=END, n=N_FILINGS,
    include_supporting=True, include_conditional=False,
    max_hits_per_form=None, metadata_path=METADATA_PATH, output_dir=OUTPUT,
)
filings_df.drop(columns='text')


0001104659-23-077401 ok 1 documents


,action,accession_number,form,filing_date,registrant_cik,registrant_name,filing_index_url,relevance,input_text_path,input_hash,document_count,status,input_coverage,error
0,exchange_offer,0001104659-23-077401,424B3,2023-07-03,0001861974,ECARX Holdings Inc.,https://www.sec.gov/Archives/edgar/data/186197...,candidate_not_verified,/Users/aeshagandhi/Workspace/BNY-Capstone-Proj...,76226380ef87c3c3d24925654ebba8bd51aa5dd8f4a860...,1,ok,selected filing and attached exhibits only; in...,


## 3. Save filing metadata

This information comes directly from EDGAR and the loader; it does not require an LLM. The table is also saved as `filing_metadata.csv`.

- `filing_id` is the SEC accession number.
- `event_id` begins as the action plus accession number. During review, assign the same event ID to related filings that describe one event.
- `form` is the SEC filing type.
- The size fields describe the combined text prepared for the LLM.


In [ ]:
FILING_METADATA_PATH = OUTPUT / "filing_metadata.csv"
metadata_columns = [
    "action", "event_id", "filing_id", "accession_number", "form", "filing_date",
    "registrant_cik", "registrant_name", "filing_index_url", "document_count",
    "text_char_count", "text_byte_count", "text_size_mb", "input_hash",
    "status", "relevance", "input_coverage", "error",
]
filing_metadata_df = filings_df[
    [column for column in metadata_columns if column in filings_df.columns]
].copy()
filing_metadata_df.to_csv(FILING_METADATA_PATH, index=False)
print(f"Saved {len(filing_metadata_df)} filing records to {FILING_METADATA_PATH}")
filing_metadata_df.head()


## 4. Check the text

Each row combines a filing and its attached exhibits. The preview below shows only the first 6,000 characters; the saved text is complete for documents successfully processed.

- `ok`: selected documents were processed.
- `partial`: some selected documents could not be read as text.
- `failed`: no document text was produced.

Check that the filing matches your action. Compare complex tables with the original. PDF/image attachments are saved but not converted to text. Documents referenced in other filings are not downloaded automatically, even when status is `ok`.


In [9]:
if not filings_df.empty:
    print(filings_df[['accession_number', 'status']].to_string(index=False))
    print(filings_df.iloc[0]['text'][:6000])
manifest_path = OUTPUT / 'documents.csv'
if manifest_path.exists():
    display(pd.read_csv(manifest_path)[['accession_number', 'filename', 'document_type', 'role', 'status', 'error']])


    accession_number status
0001104659-23-077401     ok
=== DOCUMENT tm231588-19_424b3.htm | 424B3 | https://www.sec.gov/Archives/edgar/data/1861974/000110465923077401/tm231588-19_424b3.htm ===
424B3
1
tm231588-19_424b3.htm
424B3
tm231588-19_424b3 - none - 48.2916238s
TABLE OF CONTENTS
Filed Pursuant to Rule 424(b)(3)​
Registration No. 333-271861​
PROSPECTUS
ECARX Holdings Inc.
23,871,971 CLASS A ORDINARY SHARES UNDERLYING WARRANTS,
291,679,672 CLASS A ORDINARY SHARES AND
8,872,000 WARRANTS TO PURCHASE CLASS A ORDINARY SHARES
​
This prospectus relates to the issuance by ECARX Holdings Inc. of up to 23,871,971 of its Class A ordinary shares, par value US$0.000005 per share (“Class A Ordinary Shares”), including (i) 14,999,971 Class A Ordinary Shares issuable upon the exercise of warrants to purchase Class A Ordinary Shares at an exercise price of US$11.50 per share, which were issued on December 20, 2022 (the “Closing Date”) in exchange for the public warrants of COVA Acquisition Corp. 

,accession_number,filename,document_type,role,status,error
0,0001104659-23-077401,tm231588-19_424b3.htm,424B3,primary,ok,NaN
1,0001104659-23-077401,fc_corporatestru-bw.jpg,GRAPHIC,other,excluded,NaN
2,0001104659-23-077401,0001104659-23-077401.txt,NaN,other,excluded,NaN


## 5. Use the text with your LLM

Use a reviewed row’s `text` as the filing input in `Evaluation_LLM.ipynb`. We are using one LLM for now. The example below stays commented out so it does not call StackAI.

If the text is too long for the model, split it into documented sections first; do not silently cut it off. Downloads and prepared text are saved in the `OUTPUT` folder.


In [10]:
# After checking the text, use a row in your LLM notebook:
# row = filings_df.iloc[0]
# filing_text = row['text']
# output = query({'in-0': filing_text, 'in-1': action_prompt})
# Attach row['accession_number'] and row['filing_date'] to parsed results.

# Reload saved results without downloading again:
# filings_df = pd.read_json(OUTPUT / 'filings.jsonl', lines=True, convert_dates=False)
